In [1]:
# ============================================
# Tailored Sourcing - Individual Assignment
# ============================================

import numpy as np
import pandas as pd

# Suppress scientific notation
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ============================================
# INPUT DATA
# ============================================

# Product data
products = pd.DataFrame({
    'Product': [1, 2, 3, 4],
    'Demand':  [1000, 300, 100, 50],
    'Specific_cost': [20, 25, 30, 50],
    'Unit_cost': [50, 60, 30, 30],
})

# Common parameters
S = 150       # Common ordering cost
h = 0.15      # Holding cost rate

# Derived: holding cost per unit per year
products['H'] = h * products['Unit_cost']

print("=== Input Data ===")
print(products.to_string(index=False))
print(f"\nCommon ordering cost S = ${S}")
print(f"Holding cost rate h = {h}")

=== Input Data ===
 Product  Demand  Specific_cost  Unit_cost      H
       1    1000             20         50 7.5000
       2     300             25         60 9.0000
       3     100             30         30 4.5000
       4      50             50         30 4.5000

Common ordering cost S = $150
Holding cost rate h = 0.15


In [5]:
# ============================================
# PART 1: INDEPENDENT ORDERING
# ============================================

print("=" * 50)
print("PART 1: INDEPENDENT ORDERING")
print("=" * 50)

results_ind = []

for _, row in products.iterrows():
    D = row['Demand']
    s = row['Specific_cost']
    H = row['H']

    S_total = S + s  # total fixed cost per order

    # Optimal order frequency
    n_star = np.sqrt((D * H) / (2 * S_total))

    # Annual costs (at optimum, holding = ordering)
    annual_ordering = n_star * S_total
    annual_holding  = (D * H) / (2 * n_star)
    total_cost      = annual_ordering + annual_holding

    results_ind.append({
        'Product':          int(row['Product']),
        'S_total':          S_total,
        'n* (orders/yr)':   round(n_star, 4),
        'Annual Ordering':  round(annual_ordering, 2),
        'Annual Holding':   round(annual_holding, 2),
        'Total Cost':       round(total_cost, 2)
    })

df_ind = pd.DataFrame(results_ind)
print(df_ind.to_string(index=False))
print(f"\n>>> TOTAL COST (Independent) = ${df_ind['Total Cost'].sum():,.2f}")
total_independent = df_ind['Total Cost'].sum()

PART 1: INDEPENDENT ORDERING
 Product  S_total  n* (orders/yr)  Annual Ordering  Annual Holding  Total Cost
       1 170.0000          4.6967         798.4400        798.4400   1596.8700
       2 175.0000          2.7775         486.0600        486.0600    972.1100
       3 180.0000          1.1180         201.2500        201.2500    402.4900
       4 200.0000          0.7500         150.0000        150.0000    300.0000

>>> TOTAL COST (Independent) = $3,271.47


In [6]:
# ============================================
# PART 2: COMPLETE AGGREGATION
# ============================================

print("=" * 50)
print("PART 2: COMPLETE AGGREGATION")
print("=" * 50)

# Total fixed cost per joint order
S_joint = S + products['Specific_cost'].sum()
print(f"Total fixed cost per order: S + Σs_i = {S} + {products['Specific_cost'].sum()} = ${S_joint}")

# Sum of D_i * H_i
sum_DH = (products['Demand'] * products['H']).sum()
print(f"Σ D_i * H_i = {sum_DH}")

# Optimal joint frequency
n_joint = np.sqrt(sum_DH / (2 * S_joint))
print(f"\nOptimal joint frequency n* = sqrt({sum_DH} / {2*S_joint}) = {n_joint:.4f} orders/yr")

# Annual costs
annual_ordering_joint = n_joint * S_joint
annual_holding_joint  = sum_DH / (2 * n_joint)
total_complete        = annual_ordering_joint + annual_holding_joint

print(f"\nAnnual ordering cost = {n_joint:.4f} × {S_joint} = ${annual_ordering_joint:,.2f}")
print(f"Annual holding cost  = {sum_DH} / (2 × {n_joint:.4f}) = ${annual_holding_joint:,.2f}")
print(f"\n>>> TOTAL COST (Complete Aggregation) = ${total_complete:,.2f}")

PART 2: COMPLETE AGGREGATION
Total fixed cost per order: S + Σs_i = 150 + 125 = $275
Σ D_i * H_i = 10875.0

Optimal joint frequency n* = sqrt(10875.0 / 550) = 4.4467 orders/yr

Annual ordering cost = 4.4467 × 275 = $1,222.83
Annual holding cost  = 10875.0 / (2 × 4.4467) = $1,222.83

>>> TOTAL COST (Complete Aggregation) = $2,445.66


In [11]:
# ============================================
# PART 3: TAILORED AGGREGATION
# ============================================

print("=" * 50)
print("PART 3: TAILORED AGGREGATION")
print("=" * 50)

D = products['Demand'].values
s = products['Specific_cost'].values
H = products['H'].values
n = len(products)

# ── STEP 1: Independent frequencies (full cost S + s_i) ──────────────
n_ind = np.sqrt((D * H) / (2 * (S + s)))
print("\n--- Step 1: Independent order frequencies (with common cost) ---")
for i in range(n):
    print(f"  Product {i+1}: n = sqrt({D[i]}×{H[i]} / (2×{S+s[i]})) = {n_ind[i]:.4f}")

base = np.argmax(n_ind)
print(f"\n  Most frequently ordered product: Product {base+1} (n = {n_ind[base]:.4f})")

# ── STEP 2: Recompute frequencies using only specific costs (S=0) ─────
n_specific = np.sqrt((D * H) / (2 * s))
print("\n--- Step 2: Frequencies using only specific costs (S=0) ---")
for i in range(n):
    if i != base:
        print(f"  Product {i+1}: n = sqrt({D[i]}×{H[i]} / (2×{s[i]})) = {n_specific[i]:.4f}")

# ── STEP 3: Multipliers m_i = roundup(n_base / n_specific_i) ─────────
import math
m = np.ones(n, dtype=int)
print("\n--- Step 3: Multipliers m_i = roundup(n_base / n_i_specific) ---")
for i in range(n):
    if i != base:
        ratio = n_ind[base] / n_specific[i]
        m[i]  = math.ceil(ratio)
        print(f"  Product {i+1}: m = roundup({n_ind[base]:.4f} / {n_specific[i]:.4f}) = roundup({ratio:.4f}) = {m[i]}")
    else:
        print(f"  Product {i+1}: m = 1 (base product)")

# ── STEP 4: Recalculate n* for base product ───────────────────────────
numerator   = np.sum(D * H * m)
denominator = 2 * (S + np.sum(s / m))
n_tailored  = np.sqrt(numerator / denominator)

print(f"\n--- Step 4: Recalculate n* for base product ---")
print(f"  Numerator   = Σ D_i×H_i×m_i = {numerator}")
print(f"  Denominator = 2×(S + Σs_i/m_i) = 2×({S} + {np.sum(s/m):.2f}) = {denominator:.2f}")
print(f"  n* = sqrt({numerator}/{denominator:.2f}) = {n_tailored:.4f} orders/yr")

# ── STEP 5: Order frequency per product ──────────────────────────────
n_each = n_tailored / m
print(f"\n--- Step 5: Order frequency per product ---")
for i in range(n):
    print(f"  Product {i+1}: n = {n_tailored:.4f} / {m[i]} = {n_each[i]:.4f} orders/yr")

PART 3: TAILORED AGGREGATION

--- Step 1: Independent order frequencies (with common cost) ---
  Product 1: n = sqrt(1000×7.5 / (2×170)) = 4.6967
  Product 2: n = sqrt(300×9.0 / (2×175)) = 2.7775
  Product 3: n = sqrt(100×4.5 / (2×180)) = 1.1180
  Product 4: n = sqrt(50×4.5 / (2×200)) = 0.7500

  Most frequently ordered product: Product 1 (n = 4.6967)

--- Step 2: Frequencies using only specific costs (S=0) ---
  Product 2: n = sqrt(300×9.0 / (2×25)) = 7.3485
  Product 3: n = sqrt(100×4.5 / (2×30)) = 2.7386
  Product 4: n = sqrt(50×4.5 / (2×50)) = 1.5000

--- Step 3: Multipliers m_i = roundup(n_base / n_i_specific) ---
  Product 1: m = 1 (base product)
  Product 2: m = roundup(4.6967 / 7.3485) = roundup(0.6391) = 1
  Product 3: m = roundup(4.6967 / 2.7386) = roundup(1.7150) = 2
  Product 4: m = roundup(4.6967 / 1.5000) = roundup(3.1311) = 4

--- Step 4: Recalculate n* for base product ---
  Numerator   = Σ D_i×H_i×m_i = 12000.0
  Denominator = 2×(S + Σs_i/m_i) = 2×(150 + 72.50) = 445.0

In [8]:
# ── Annual costs ──────────────────────────────────────────────────────
annual_ordering_tailored = n_tailored * (S + np.sum(s / m))
annual_holding_tailored  = np.sum((D * H) / (2 * n_each))
total_tailored           = annual_ordering_tailored + annual_holding_tailored

print("\n--- Annual Costs (Tailored Aggregation) ---")
print(f"  Annual ordering cost = {n_tailored:.4f} × {S + np.sum(s/m):.2f} = ${annual_ordering_tailored:,.2f}")
print(f"  Annual holding cost  = Σ D_i×H_i / (2×n_i)            = ${annual_holding_tailored:,.2f}")
print(f"\n>>> TOTAL COST (Tailored Aggregation) = ${total_tailored:,.2f}")

# Multipliers summary table
df_tailored = pd.DataFrame({
    'Product':            products['Product'].values,
    'Demand':             D,
    'm_i':                m,
    'n_i (orders/yr)':    np.round(n_each, 4),
    'Annual Holding':     np.round((D * H) / (2 * n_each), 2),
})
print("\n--- Per-Product Breakdown ---")
print(df_tailored.to_string(index=False))


--- Annual Costs (Tailored Aggregation) ---
  Annual ordering cost = 5.1929 × 222.50 = $1,155.42
  Annual holding cost  = Σ D_i×H_i / (2×n_i)            = $1,155.42

>>> TOTAL COST (Tailored Aggregation) = $2,310.84

--- Per-Product Breakdown ---
 Product  Demand  m_i  n_i (orders/yr)  Annual Holding
       1    1000    1           5.1929        722.1400
       2     300    1           5.1929        259.9700
       3     100    2           2.5965         86.6600
       4      50    4           1.2982         86.6600


In [10]:
# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "=" * 50)
print("FINAL SUMMARY")
print("=" * 50)

summary = pd.DataFrame({
    'Strategy': [
        '1. Independent Ordering',
        '2. Complete Aggregation',
        '3. Tailored Aggregation'
    ],
    'Annual Cost ($)': [
        round(total_independent, 2),
        round(total_complete, 2),
        round(total_tailored, 2)
    ]
})

print(summary.to_string(index=False))
print(f"\n Optimal strategy: Tailored Aggregation")
print(f" Lowest annual cost: ${total_tailored:,.2f}")


FINAL SUMMARY
               Strategy  Annual Cost ($)
1. Independent Ordering        3271.4700
2. Complete Aggregation        2445.6600
3. Tailored Aggregation        2310.8400

 Optimal strategy: Tailored Aggregation
 Lowest annual cost: $2,310.84
